In [1]:
from datetime import date

import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine



# database connections 

In [2]:
with open('../config_fill.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['CO_SA']
    config_etl = config['ETL_PRO']

url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
co_sa = create_engine(url_co)
etl_conn = create_engine(url_etl)

# Extract


In [3]:
tabla_fases = pd.read_sql_table('mensajeria_estadosservicio', co_sa)
servicios   = pd.read_sql_table('mensajeria_servicio', co_sa)
dim_sede    = pd.read_sql_table('dim_sede', etl_conn)
dim_fecha   = pd.read_sql_table('dim_fecha', etl_conn)
tabla_fases.info()

<class 'pandas.DataFrame'>
RangeIndex: 128402 entries, 0 to 128401
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype        
---  ------         --------------   -----        
 0   id             128402 non-null  int64        
 1   fecha          128402 non-null  datetime64[s]
 2   hora           128402 non-null  object       
 3   foto           128402 non-null  str          
 4   observaciones  128401 non-null  str          
 5   estado_id      128402 non-null  int64        
 6   servicio_id    128402 non-null  int64        
 7   es_prueba      128402 non-null  bool         
 8   foto_binary    270 non-null     object       
dtypes: bool(1), datetime64[s](1), int64(3), object(2), str(2)
memory usage: 8.0+ MB


# Transformations


In [4]:
hecho = tabla_fases.copy()
hecho = hecho[['id', 'fecha', 'hora', 'estado_id', 'servicio_id']]


hecho['ts'] = pd.to_datetime(
    hecho['fecha'].astype(str) + ' ' + hecho['hora'].astype(str),
    format='mixed'
)


hecho = hecho.sort_values(['servicio_id', 'ts'])
hecho['ts_siguiente'] = hecho.groupby('servicio_id')['ts'].shift(-1)
hecho['tiempo_espera_min'] = (hecho['ts_siguiente'] - hecho['ts']).dt.total_seconds() / 60

hecho.info()

<class 'pandas.DataFrame'>
Index: 128402 entries, 301 to 128394
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   id                 128402 non-null  int64         
 1   fecha              128402 non-null  datetime64[s] 
 2   hora               128402 non-null  object        
 3   estado_id          128402 non-null  int64         
 4   servicio_id        128402 non-null  int64         
 5   ts                 128402 non-null  datetime64[us]
 6   ts_siguiente       99972 non-null   datetime64[us]
 7   tiempo_espera_min  99972 non-null   float64       
dtypes: datetime64[s](1), datetime64[us](2), float64(1), int64(3), object(1)
memory usage: 8.8+ MB


In [5]:
servicios_min = servicios[['id', 'cliente_id', 'mensajero_id']]
hecho = hecho.merge(servicios_min, left_on='servicio_id', right_on='id', how='left')

sede_por_cliente = dim_sede[['cliente_id', 'sede_id']].drop_duplicates(subset='cliente_id')
hecho = hecho.merge(sede_por_cliente, on='cliente_id', how='left')

hecho['id_hora'] = hecho['ts'].dt.hour * 60 + hecho['ts'].dt.minute

dim_fecha_min = dim_fecha[['key_dim_fecha', 'fecha_completa']].copy()
dim_fecha_min['fecha_completa'] = pd.to_datetime(dim_fecha_min['fecha_completa']).dt.date


hecho['fecha'] = pd.to_datetime(hecho['fecha']).dt.date

hecho = hecho.merge(dim_fecha_min, left_on='fecha', right_on='fecha_completa', how='left')
hecho.info()

<class 'pandas.DataFrame'>
RangeIndex: 128402 entries, 0 to 128401
Data columns (total 15 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   id_x               128402 non-null  int64         
 1   fecha              128402 non-null  object        
 2   hora               128402 non-null  object        
 3   estado_id          128402 non-null  int64         
 4   servicio_id        128402 non-null  int64         
 5   ts                 128402 non-null  datetime64[us]
 6   ts_siguiente       99972 non-null   datetime64[us]
 7   tiempo_espera_min  99972 non-null   float64       
 8   id_y               128402 non-null  int64         
 9   cliente_id         128402 non-null  int64         
 10  mensajero_id       127675 non-null  float64       
 11  sede_id            128402 non-null  int64         
 12  id_hora            128402 non-null  int32         
 13  key_dim_fecha      128402 non-null  int64         
 14 

In [6]:
hecho_seguimiento_fases = hecho[[
    'key_dim_fecha', 'id_hora', 'cliente_id', 'sede_id',
    'mensajero_id', 'estado_id', 'servicio_id', 'tiempo_espera_min'
]]

hecho_seguimiento_fases.info()
hecho_seguimiento_fases.head()

<class 'pandas.DataFrame'>
RangeIndex: 128402 entries, 0 to 128401
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   key_dim_fecha      128402 non-null  int64  
 1   id_hora            128402 non-null  int32  
 2   cliente_id         128402 non-null  int64  
 3   sede_id            128402 non-null  int64  
 4   mensajero_id       127675 non-null  float64
 5   estado_id          128402 non-null  int64  
 6   servicio_id        128402 non-null  int64  
 7   tiempo_espera_min  99972 non-null   float64
dtypes: float64(2), int32(1), int64(5)
memory usage: 7.3 MB


,key_dim_fecha,id_hora,cliente_id,sede_id,mensajero_id,estado_id,servicio_id,tiempo_espera_min
0,3184,982,5,5,1.0,1,7,34649.033333
1,3208,1071,5,5,1.0,2,7,25571.480733
2,3226,722,5,5,1.0,4,7,13.185933
3,3226,736,5,5,1.0,6,7,291.916667
4,3226,1027,5,5,1.0,5,7,NaN


# load

In [7]:
hecho_seguimiento_fases.to_sql('hecho_seguimiento_fases', etl_conn, if_exists='replace', index_label='key_hecho_seguimiento_fases')

402